# K0 Random Run 07 — 清理版源运行 Notebook

这个 notebook 对应 **k0-random-run-07** 的源运行流程。

核心原则：

- 在关键随机步骤前保存当前 NumPy RNG state；
- 保留最终 K0 pipeline 必需代码；
- 输出 submission 和复现包。

三个保存点：

1. shuffle 前；
2. SMOTE 前；
3. final XGBoost 训练前。

In [20]:
# ===== 1. 导入必要库 =====
# 本单元只负责“准备工具”，不做任何数据处理，也不训练模型。
# 下面这些 import 都是后续 K0 源运行流程真正会用到的库。

# os / sys：读取系统路径和 Python 版本信息。
import os
import sys

# json：把环境信息保存成 json 文件，方便之后确认运行环境。
import json

# time：生成 TAG 时间戳，用来给本次输出文件命名，避免覆盖旧文件。
import time

# pickle：保存 NumPy RNG state，也就是随机数生成器的内部状态。
import pickle

# zipfile：把 submission、RNG state 和环境信息打包成 zip。
import zipfile

# platform：记录操作系统 / 平台信息。
import platform

# pathlib.Path：更稳定地处理文件路径。
from pathlib import Path

# numpy：数值计算，同时也是本 notebook 要保存 RNG state 的来源。
import numpy as np

# pandas：读取 csv、处理表格数据。
import pandas as pd

# SimpleImputer：对缺失值做填补。
from sklearn.impute import SimpleImputer

# OneHotEncoder：把类别变量转换成模型能读的 0/1 特征。
from sklearn.preprocessing import OneHotEncoder

# shuffle：打乱训练数据顺序，这是第一个需要保存 RNG state 的随机步骤。
from sklearn.utils import shuffle

# accuracy_score / StratifiedKFold / cross_val_score：保留原流程中可能使用的评估工具。
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

# SMOTE：对少数类进行合成过采样，这是第二个需要保存 RNG state 的随机步骤。
from imblearn.over_sampling import SMOTE

# xgboost：最终主模型使用 XGBoost 分类器。
import xgboost as xgb

# 显示所有列，方便检查中间表格。
pd.set_option("display.max_columns", None)

# 如果在 Kaggle 环境运行，就把输出目录设为 /kaggle/working。
# 如果在本地运行，就把输出目录设为当前目录。
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

# TAG 是当前时间戳。后面所有输出文件都会带这个 TAG，避免文件名冲突。
TAG = int(time.time())

# 打印本次运行的基本信息，方便确认环境。
print("TAG:", TAG)
print("Output directory:", OUT)
print("Python:", sys.version)
print("Platform:", platform.platform())


TAG: 1779957904
Output directory: /kaggle/working
Python: 3.7.12 | packaged by conda-forge | (default, Oct 26 2021, 06:08:53) 
[GCC 9.4.0]
Platform: Linux-6.6.122+-x86_64-with-debian-bullseye-sid


In [21]:
# ===== 2. 保存环境信息 =====
# 这个单元只做记录，不影响模型结果。
# 目的：把本次运行环境保存下来，之后复现时可以对比 Python / package 版本。

# env_info 是一个字典，用来保存本次运行的环境信息。
env_info = {
    "tag": TAG,                         # 本次运行的时间戳 ID
    "python": sys.version,              # Python 版本
    "platform": platform.platform(),    # 操作系统 / 平台信息
    "numpy": np.__version__,            # NumPy 版本
    "pandas": pd.__version__,           # pandas 版本
}

try:
    # 这些库的版本也记录下来，因为模型结果可能受版本影响。
    import sklearn
    import imblearn
    import xgboost

    # 保存 sklearn / imblearn / xgboost 版本。
    env_info["sklearn"] = sklearn.__version__
    env_info["imblearn"] = imblearn.__version__
    env_info["xgboost"] = xgboost.__version__

except Exception as e:
    # 如果某个版本读取失败，就记录错误信息，但不中断整个流程。
    env_info["version_error"] = str(e)

# 环境信息输出路径。
env_path = OUT / f"env_info_{TAG}.json"

# 把 env_info 写入 json 文件。
with open(env_path, "w", encoding="utf-8") as f:
    json.dump(env_info, f, indent=2)

# 打印保存路径，并显示 env_info 内容。
print("Saved env info:", env_path)
env_info


Saved env info: /kaggle/working/env_info_1779957904.json


{'tag': 1779957904,
 'python': '3.7.12 | packaged by conda-forge | (default, Oct 26 2021, 06:08:53) \n[GCC 9.4.0]',
 'platform': 'Linux-6.6.122+-x86_64-with-debian-bullseye-sid',
 'numpy': '1.21.6',
 'pandas': '1.3.5',
 'sklearn': '1.0.2',
 'imblearn': '0.9.0',
 'xgboost': '1.6.1'}

In [22]:
# ===== 3. 读取 Kaggle 官方数据 =====
# train.csv：训练集，包含 Transported 标签。
# test.csv：测试集，没有 Transported 标签，需要我们预测。
# sample_submission.csv：Kaggle 提交模板，最后要按这个格式生成 submission。

# 可能的数据目录。Kaggle 官方数据通常在 /kaggle/input/spaceship-titanic。
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/spaceship-titanic"),
    Path("/kaggle/input"),
    Path("."),
]

# 这个函数用于在多个目录里寻找指定文件。
def find_file(filename, roots):
    # 遍历所有候选根目录。
    for root in roots:
        # 如果这个目录不存在，就跳过。
        if not root.exists():
            continue

        # 先检查 root/filename 是否直接存在。
        direct = root / filename
        if direct.exists():
            return direct

        # 如果不在根目录下，就递归搜索子目录。
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]

    # 所有位置都找不到时，返回 None。
    return None

# 分别寻找 train/test/sample_submission 三个文件。
train_path = find_file("train.csv", DATA_ROOT_CANDIDATES)
test_path = find_file("test.csv", DATA_ROOT_CANDIDATES)
sample_path = find_file("sample_submission.csv", DATA_ROOT_CANDIDATES)

# 如果找不到任何一个关键文件，直接报错停止。
assert train_path is not None, "找不到 train.csv"
assert test_path is not None, "找不到 test.csv"
assert sample_path is not None, "找不到 sample_submission.csv"

# 读取 csv 文件为 pandas DataFrame。
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

# 打印数据形状，确认读取成功。
print("train:", train.shape, train_path)
print("test:", test.shape, test_path)
print("sample:", sample.shape, sample_path)

# 查看训练集前几行。
train.head()


train: (8693, 14) /kaggle/input/spaceship-titanic/train.csv
test: (4277, 13) /kaggle/input/spaceship-titanic/test.csv
sample: (4277, 2) /kaggle/input/spaceship-titanic/sample_submission.csv


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## 4. 数据预处理与特征重建

这一部分保留 run-07 的最终有效预处理逻辑：

1. CryoSleep 与消费规则；
2. Expenses 总消费；
3. RoomFill：用同组乘客的非标签信息补缺失；
4. Cabin 拆分；
5. SimpleImputer + OneHotEncoder。

In [23]:
# ===== 4.1 合并 train 和 test 做非标签字段预处理 =====
# 这一步是把 train 和 test 上下拼接成一张大表 train_test。
# 目的：让训练集和测试集走同一套非标签预处理流程。
# 注意：test 没有 Transported 标签；后续也不会用 test 的标签信息训练模型。

try:
    # 原 notebook 使用 append。这里优先保留原写法。
    # append 会把 test 追加到 train 后面。
    train_test = train.append(test)

except AttributeError:
    # pandas 2.x 删除了 DataFrame.append。
    # pd.concat([train, test], axis=0) 是等价写法：按行拼接。
    # 这里不 reset_index，是为了尽量保留原 notebook 的索引语义。
    train_test = pd.concat([train, test], axis=0)

# 打印合并后的行数和列数。
print("Combined shape:", train_test.shape)

# 打印每一列的缺失值数量，方便确认后续需要处理哪些列。
print(train_test.isnull().sum())


Combined shape: (12970, 14)
PassengerId        0
HomePlanet       288
CryoSleep        310
Cabin            299
Destination      274
Age              270
VIP              296
RoomService      263
FoodCourt        289
ShoppingMall     306
Spa              284
VRDeck           268
Name             294
Transported     4277
dtype: int64


In [24]:
# ===== 4.2 CryoSleep 与消费规则 =====
# 这一步利用 CryoSleep 和消费之间的物理规则做特征修正。
# 规则逻辑：
# 1. 如果 CryoSleep=True，乘客处于冷冻休眠状态，五项消费应为 0。
# 2. Expenses 是五项消费总和。
# 3. 如果 CryoSleep 缺失，并且 Expenses=0，则推断 CryoSleep=True。

# 五项消费列。
Expenses_columns = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# 对每一行应用规则：
# 如果这一行 CryoSleep=True，则该行五项消费写成 0；
# 否则保持原行不变。
# 这里保留原 run-07 写法，不改变逻辑。
train_test.loc[:, Expenses_columns] = train_test.apply(
    lambda x: 0 if x.CryoSleep == True else x,
    axis=1
)

# Expenses = 五项消费总和。
train_test["Expenses"] = train_test.loc[:, Expenses_columns].sum(axis=1)

# 如果 CryoSleep 缺失，并且总消费为 0，则推断 CryoSleep=True。
# 这里同样保留原 run-07 写法。
train_test.loc[:, ["CryoSleep"]] = train_test.apply(
    lambda x: True if x.Expenses == 0 and pd.isna(x.CryoSleep) else x,
    axis=1
)

# 检查 CryoSleep 和 Expenses 的缺失情况。
print(train_test[["CryoSleep", "Expenses"]].isnull().sum())


CryoSleep    174
Expenses       0
dtype: int64


In [25]:
# ===== 4.3 Name 缺失处理 =====
# Name 后面会被拆成 FirstName / SecondName，并构造 Name_key。
# 虽然最终模型没有直接使用 Name_key，但为了保留原 run-07 的处理顺序，这里仍然执行。

# 把缺失的姓名统一填成 "Unknown Unknown"。
# 这样后面 split(" ") 时不会因为缺失值报错。
train_test.Name = train_test.Name.fillna("Unknown Unknown")


In [26]:
# ===== 4.4 RoomFill：用同组乘客信息补缺失 =====
# PassengerId 的前四位作为 Room / group prefix。
# 同组乘客往往共享 Cabin、VIP、HomePlanet、Destination 等非标签信息。
# 本步骤只使用非标签字段，不使用 Transported 标签。

# 提取 PassengerId 前四位，作为 Room/group 编号。
train_test.loc[:, ["Room"]] = train_test.PassengerId.apply(lambda x: x[0:4])

# 为每个 Room 建立 VIP 参考表：
# 只保留 Room 和 VIP 两列，去掉缺失值，每个 Room 只保留第一条已知记录。
guide_VIP = train_test.loc[:, ["Room", "VIP"]].dropna().drop_duplicates("Room")

# 为每个 Room 建立 Cabin 参考表。
guide_Cabin = train_test.loc[:, ["Room", "Cabin"]].dropna().drop_duplicates("Room")

# 为每个 Room 建立 HomePlanet 参考表。
guide_HomePlanet = train_test.loc[:, ["Room", "HomePlanet"]].dropna().drop_duplicates("Room")

# 为每个 Room 建立 Destination 参考表。
guide_Destination = train_test.loc[:, ["Room", "Destination"]].dropna().drop_duplicates("Room")

# 把 Room 对应的 Cabin 参考值合并回 train_test。
# 如果原列已有值，后面会优先保留原值；如果原列缺失，才用 _y 参考列补。
train_test = pd.merge(train_test, guide_Cabin, how="left", on="Room", suffixes=("", "_y"))

# 合并 VIP 参考值。
train_test = pd.merge(train_test, guide_VIP, how="left", on="Room", suffixes=("", "_y"))

# 合并 HomePlanet 参考值。
train_test = pd.merge(train_test, guide_HomePlanet, how="left", on="Room", suffixes=("", "_y"))

# 合并 Destination 参考值。
train_test = pd.merge(train_test, guide_Destination, how="left", on="Room", suffixes=("", "_y"))

# 如果 VIP 缺失，则用同 Room 的 VIP_y 补；否则保留原 VIP。
train_test.loc[:, ["VIP"]] = train_test.VIP.fillna(train_test.VIP_y)

# 如果 Cabin 缺失，则用同 Room 的 Cabin_y 补；否则保留原 Cabin。
train_test.loc[:, ["Cabin"]] = train_test.Cabin.fillna(train_test.Cabin_y)

# 如果 HomePlanet 缺失，则用同 Room 的 HomePlanet_y 补；否则保留原 HomePlanet。
train_test.loc[:, ["HomePlanet"]] = train_test.HomePlanet.fillna(train_test.HomePlanet_y)

# 如果 Destination 缺失，则用同 Room 的 Destination_y 补；否则保留原 Destination。
train_test.loc[:, ["Destination"]] = train_test.Destination.fillna(train_test.Destination_y)

# 删除临时参考列，避免进入后续特征。
train_test = train_test.drop(["VIP_y", "Cabin_y", "HomePlanet_y", "Destination_y"], axis=1)

# 查看 RoomFill 后关键列剩余缺失数量。
print(train_test[["VIP", "Cabin", "HomePlanet", "Destination"]].isnull().sum())


VIP            172
Cabin          162
HomePlanet     157
Destination    154
dtype: int64


In [27]:
# ===== 4.5 拆分 Cabin 和 Name =====
# Cabin 原格式类似 "B/0/P"。
# 拆分后：
# Cabin_1：Deck，表示舱位甲板；
# Cabin_2：Cabin number，表示舱号；
# Cabin_3：Side，表示左右舷。
# 最终 run-07 使用 Cabin_1 和 Cabin_3，不使用 Cabin_2。

# 拆出 Cabin_1，也就是 Deck。
train_test.loc[:, ["Cabin_1"]] = train_test.Cabin.str.split("/", expand=True).iloc[:, 0]

# 拆出 Cabin_2，也就是 Cabin number。
train_test.loc[:, ["Cabin_2"]] = train_test.Cabin.str.split("/", expand=True).iloc[:, 1]

# 拆出 Cabin_3，也就是 Side。
train_test.loc[:, ["Cabin_3"]] = train_test.Cabin.str.split("/", expand=True).iloc[:, 2]

# 把 Name 拆成 FirstName。
train_test.loc[:, ["FirstName"]] = train_test.Name.str.split(" ", expand=True).iloc[:, 0]

# 把 Name 拆成 SecondName。
train_test.loc[:, ["SecondName"]] = train_test.Name.str.split(" ", expand=True).iloc[:, 1]

# 构造 Name_key。保留原流程中的构造顺序。
train_test["Name_key"] = train_test["SecondName"] + train_test["Room"]


In [28]:
# ===== 4.6 SimpleImputer + OneHotEncoder =====
# 这一段把表格整理成 XGBoost 可以直接训练的数值矩阵。
# 数值列：用均值填补缺失值。
# 类别列：用众数填补缺失值。
# 类别编码：用 OneHotEncoder 转成 0/1 特征。

# 最终进入模型的数值列。
num_cols = ["ShoppingMall", "FoodCourt", "RoomService", "Spa", "VRDeck", "Expenses", "Age"]

# 最终进入模型的类别列。
cat_cols = ["CryoSleep", "Cabin_1", "Cabin_3", "VIP", "HomePlanet", "Destination"]

# 标签列。train 有值，test 为空。
transported = ["Transported"]

# 只保留最终模型需要的列，避免无关列进入模型。
train_test = train_test[num_cols + cat_cols + transported].copy()

# 数值列缺失值填补器：均值填补。
num_imp = SimpleImputer(strategy="mean")

# 类别列缺失值填补器：众数填补。
cat_imp = SimpleImputer(strategy="most_frequent")

try:
    # 旧版 scikit-learn 使用 sparse=False。
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
except TypeError:
    # 新版 scikit-learn 使用 sparse_output=False。
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# 对数值列做均值填补。
# fit_transform 会先学习每列均值，再把缺失值替换掉。
train_test[num_cols] = pd.DataFrame(
    num_imp.fit_transform(train_test[num_cols]),
    columns=num_cols
)

# 对类别列做众数填补。
train_test[cat_cols] = pd.DataFrame(
    cat_imp.fit_transform(train_test[cat_cols]),
    columns=cat_cols
)

# 对类别列做 OneHot 编码。
# 例如 HomePlanet 会变成 HomePlanet_Earth / HomePlanet_Europa / HomePlanet_Mars 等 0/1 列。
temp_train = pd.DataFrame(
    ohe.fit_transform(train_test[cat_cols]),
    columns=ohe.get_feature_names_out()
)

# 拼接数值列、OneHot 后的类别列、Transported 标签列。
train_test = train_test[num_cols + transported].join(temp_train)

# 查看处理后的表格大小。
print("Processed train_test:", train_test.shape)

# 查看前几行，确认特征已经全部变成模型可用的数值形式。
train_test.head()


Processed train_test: (12970, 28)


,ShoppingMall,FoodCourt,RoomService,Spa,VRDeck,Expenses,Age,Transported,CryoSleep_False,CryoSleep_True,Cabin_1_A,Cabin_1_B,Cabin_1_C,Cabin_1_D,Cabin_1_E,Cabin_1_F,Cabin_1_G,Cabin_1_T,Cabin_3_P,Cabin_3_S,VIP_False,VIP_True,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,0.0,0.0,0.0,0.0,0.0,0.0,39.0,False,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,25.0,9.0,109.0,549.0,44.0,736.0,24.0,True,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,0.0,3576.0,43.0,6715.0,49.0,10383.0,58.0,False,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
3,371.0,1283.0,0.0,3329.0,193.0,5176.0,33.0,False,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,151.0,70.0,303.0,565.0,2.0,1091.0,16.0,True,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [29]:
# ===== 4.7 拆回训练集和测试集 =====
# 合并预处理完成后，需要重新拆回 train 和 test。
# train：Transported 非空，用于训练。
# test：Transported 为空，用于最终预测。

# 取出 Transported 非空的行，作为训练集。
train_processed = train_test[train_test["Transported"].notnull()].copy()

# 把标签 Transported 转成 int，True/False 变成 1/0。
train_processed.Transported = train_processed.Transported.astype("int")

# 取出 Transported 为空的行，作为测试集。
# 测试集没有标签，所以 drop 掉 Transported 列。
test_processed = train_test[train_test["Transported"].isnull()].drop("Transported", axis=1)

# X 是训练特征。
X = train_processed.drop("Transported", axis=1)

# y 是训练标签。
y = train_processed.Transported

# 打印训练特征、标签、测试特征的形状。
print("X:", X.shape)
print("y:", y.shape)
print("test:", test_processed.shape)

# 查看标签分布，用于确认是否需要 SMOTE 平衡。
print(y.value_counts())


X: (8693, 27)
y: (8693,)
test: (4277, 27)
1    4378
0    4315
Name: Transported, dtype: int64


## 5. 保存 shuffle 前 RNG state，然后执行 shuffle

这是 run-07 的第一个关键随机点。

此 notebook 是源运行版：  
这里保存当前 RNG state，而不是加载已有 state。

In [30]:
# ===== 5.1 保存 shuffle 前 NumPy RNG state =====
# 这是第一个关键随机点。
# 本 notebook 是源运行版，所以这里保存“当前状态”，不是加载历史状态。
# 保存这个状态后，之后的复现 notebook 可以从这里恢复并重放 shuffle。

# 生成 shuffle 前 RNG state 文件路径。
state_before_shuffle_path = OUT / f"rng_state_before_shuffle_{TAG}.pkl"

# np.random.get_state() 会读取 NumPy 随机数生成器当前的内部状态。
# pickle.dump 把这个状态写入 pkl 文件。
with open(state_before_shuffle_path, "wb") as f:
    pickle.dump(np.random.get_state(), f)

# 打印保存位置。
print("Saved state before shuffle:", state_before_shuffle_path)

# shuffle 打乱训练数据行顺序。
# 它会消耗随机数，因此必须在 shuffle 之前保存 RNG state。
X, y = shuffle(X, y)

# 打乱后重置 X 的行索引，避免旧索引影响后续对齐。
X = X.reset_index(drop=True)

# 打乱后重置 y 的行索引，保证和 X 对齐。
y = y.reset_index(drop=True)

# 打印 shuffle 后的形状。
print("After shuffle:", X.shape, y.shape)


Saved state before shuffle: /kaggle/working/rng_state_before_shuffle_1779957904.pkl
After shuffle: (8693, 27) (8693,)


In [31]:
# ===== 5.2 删除低价值 / 冗余特征 =====
# drop_list 来自原 run-07 pipeline 的特征裁剪结果。
# 目的：删除重复信息、低价值信息和可能增加噪声的特征。
# 注意：训练集 X 和测试集 test_processed 必须删除同样的列，保证两边特征一致。

# 需要删除的特征列表。
drop_list = [
    "ShoppingMall",
    "Age",
    "CryoSleep_True",
    "HomePlanet_Earth",
    "HomePlanet_Europa",
    "VIP_True",
    "HomePlanet_Mars",
    "Destination_PSO J318.5-22",
    "VIP_False",
    "Destination_55 Cancri e",
    "FoodCourt",
    "Destination_TRAPPIST-1e",
]

# 从训练特征中删除这些列。
X = X.drop(drop_list, axis=1)

# 从测试特征中删除同样的列。
test_processed = test_processed.drop(drop_list, axis=1)

# 打印裁剪后的形状。
print("X after pruning:", X.shape)
print("test after pruning:", test_processed.shape)


X after pruning: (8693, 15)
test after pruning: (4277, 15)


## 6. 保存 SMOTE 前 RNG state，然后执行 SMOTE

SMOTE 会在少数类样本之间合成新样本。  
这是 run-07 的第二个关键随机点。

In [32]:
# ===== 6.1 保存 SMOTE 前 NumPy RNG state =====
# 这是第二个关键随机点。
# SMOTE 会根据少数类样本的邻居合成新样本，其中邻居选择/样本生成涉及随机过程。
# 因此，必须在 SMOTE 前保存 RNG state。

# 生成 SMOTE 前 RNG state 文件路径。
state_before_smote_path = OUT / f"rng_state_before_smote_{TAG}.pkl"

# 保存当前 NumPy RNG state。
with open(state_before_smote_path, "wb") as f:
    pickle.dump(np.random.get_state(), f)

# 打印保存位置。
print("Saved state before SMOTE:", state_before_smote_path)

try:
    # 旧版 imbalanced-learn 支持 n_jobs 参数。
    smote = SMOTE(sampling_strategy=1, n_jobs=-1)
except TypeError:
    # 新版可能不支持 n_jobs，因此使用兼容写法。
    smote = SMOTE(sampling_strategy=1)

# 对训练集执行 SMOTE。
# fit_resample 会生成少数类合成样本，使两类数量平衡。
X_sm, y_sm = smote.fit_resample(X, y)

# 用 SMOTE 后的新训练特征替换原 X。
X = X_sm

# 用 SMOTE 后的新训练标签替换原 y。
y = y_sm

# 打印 SMOTE 后的标签分布和特征形状。
print("After SMOTE:")
print(y.value_counts())
print("X:", X.shape)


Saved state before SMOTE: /kaggle/working/rng_state_before_smote_1779957904.pkl
After SMOTE:
0    4378
1    4378
Name: Transported, dtype: int64
X: (8756, 15)


## 7. 最终 XGBoost 模型

最终主模型是 XGBoost。  
这里使用 run-07 最终 submission 位置的参数。  
训练前保存第三个 RNG state。

In [33]:
# ===== 7.1 最终 XGBoost 参数 =====
# 这里定义 run-07 最终提交使用的 XGBoost 参数。
# XGBoost 是最终主模型，适合处理这种结构化表格特征。
# n_estimators=730 是最终训练使用的树数量。

# params_XGB_best 保存最终 XGBoost 参数。
params_XGB_best = {
    "lambda": 3.0610042624477543,          # L2 正则化，控制模型复杂度
    "alpha": 4.581902571574289,            # L1 正则化，帮助减少不重要特征影响
    "colsample_bytree": 0.9241969052729379,# 每棵树随机采样的特征比例
    "subsample": 0.9527591724824661,       # 每棵树随机采样的样本比例
    "learning_rate": 0.06672065863100594,  # 学习率，控制每棵树的更新幅度
    "n_estimators": 730,                   # 树的数量
    "max_depth": 5,                        # 每棵树最大深度
    "min_child_weight": 1,                 # 子节点最小样本权重
    "num_parallel_tree": 1,                # 每轮并行树数量
}

# 显示参数，方便检查。
params_XGB_best


{'lambda': 3.0610042624477543,
 'alpha': 4.581902571574289,
 'colsample_bytree': 0.9241969052729379,
 'subsample': 0.9527591724824661,
 'learning_rate': 0.06672065863100594,
 'n_estimators': 730,
 'max_depth': 5,
 'min_child_weight': 1,
 'num_parallel_tree': 1}

In [34]:
# ===== 7.2 保存 final model 前 RNG state，训练 XGBoost 并生成 submission =====
# 这是第三个关键随机点。
# XGBoost 训练中存在随机采样，例如 subsample 和 colsample_bytree。
# 因此，训练最终模型前需要保存 RNG state。

# 生成 final model 前 RNG state 文件路径。
state_before_final_model_path = OUT / f"rng_state_before_final_model_{TAG}.pkl"

# 保存当前 NumPy RNG state。
with open(state_before_final_model_path, "wb") as f:
    pickle.dump(np.random.get_state(), f)

# 打印保存位置。
print("Saved state before final model:", state_before_final_model_path)

# 用最终参数创建 XGBoost 分类器。
model = xgb.XGBClassifier(**params_XGB_best)

# 使用 SMOTE 后的训练数据训练模型。
model.fit(X, y)

# 对 Kaggle test_processed 进行预测。
pred_XGB_best = model.predict(test_processed)

# 复制 sample_submission，保证提交格式正确。
submission = sample.copy()

# 写入预测结果。
submission["Transported"] = pred_XGB_best

# 转成布尔类型 True/False，符合 Kaggle 提交要求。
submission["Transported"] = submission["Transported"] > 0.5

# 生成 submission 文件路径。
submission_path = OUT / f"k0_random_run_07_clean_submission_{TAG}.csv"

# 保存 submission.csv。
submission.to_csv(submission_path, index=False)

# 打印保存路径，并展示前几行。
print("Saved submission:", submission_path)
submission.head()


Saved state before final model: /kaggle/working/rng_state_before_final_model_1779957904.pkl
Saved submission: /kaggle/working/k0_random_run_07_clean_submission_1779957904.csv


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [35]:
# ===== 8. 打包输出文件 =====
# 这个 zip 是本次源运行的完整复现包。
# 里面包含：
# 1. 环境信息；
# 2. shuffle 前 RNG state；
# 3. SMOTE 前 RNG state；
# 4. final XGBoost 前 RNG state；
# 5. 本次生成的 submission。
# 后续复现 notebook 可以读取这个 zip 里的 state 文件，重放本次运行。

# 生成 zip 输出路径。
zip_path = OUT / f"k0_random_run_07_clean_package_{TAG}.zip"

# 创建 zip 文件。
with zipfile.ZipFile(zip_path, "w") as z:
    # 写入环境信息 json。
    z.write(env_path, arcname=env_path.name)

    # 写入 shuffle 前 RNG state。
    z.write(state_before_shuffle_path, arcname=state_before_shuffle_path.name)

    # 写入 SMOTE 前 RNG state。
    z.write(state_before_smote_path, arcname=state_before_smote_path.name)

    # 写入 final model 前 RNG state。
    z.write(state_before_final_model_path, arcname=state_before_final_model_path.name)

    # 写入本次 submission。
    z.write(submission_path, arcname=submission_path.name)

# 检查所有关键文件是否真的存在。
assert env_path.exists()
assert state_before_shuffle_path.exists()
assert state_before_smote_path.exists()
assert state_before_final_model_path.exists()
assert submission_path.exists()
assert zip_path.exists()

# 打印 zip 保存路径。
print("Saved package:", zip_path)

# 打印输出目录中的所有文件，方便确认 zip 和 submission 已生成。
print("\nFiles in output directory:")
for p in OUT.iterdir():
    try:
        print(p.name, p.stat().st_size)
    except Exception:
        print(p.name)


Saved package: /kaggle/working/k0_random_run_07_clean_package_1779957904.zip

Files in output directory:
rng_state_before_final_model_1779954951.pkl 2685
rng_state_before_final_model_1779957904.pkl 2684
env_info_1779954951.json 293
rng_state_before_shuffle_1779954951.pkl 2685
env_info_1779957904.json 293
k0_random_run_07_clean_package_1779957904.zip 66746
k0_random_run_07_clean_submission_1779954951.csv 57625
.virtual_documents 4096
rng_state_before_smote_1779957904.pkl 2685
rng_state_before_shuffle_1779957904.pkl 2685
k0_random_run_07_clean_package_1779954951.zip 66757
k0_random_run_07_clean_submission_1779957904.csv 57615
run07_assets_extracted 4096
rng_state_before_smote_1779954951.pkl 2685


## 结果说明

这个 notebook 是 **k0-random-run-07 的清理版源运行流程**。

它做的是：

- 从当前运行环境开始；
- 执行 K0 pipeline；
- 在三个关键随机点保存 RNG state；
- 生成 submission；
- 打包保存本次 run 的状态文件。

它不从外部加载已有 RNG state。  
如果需要精确复现某一次历史 submission，应使用对应的复现 notebook 和该次 run 保存的 RNG state 文件。